# GDACS earthquakes + tropical cyclones overlay

Pull **earthquakes** (`EQ`) and **tropical cyclones** (`TC`) for a recent window over the Asia-Pacific region, overlay them on one map (distinct markers per hazard, coloured by alert level), and show how the `glide` column cross-references each alert to ReliefWeb / Copernicus EMS.

Both hazard types come back in one request — `variables=['EQ', 'TC']`. GDACS has no server-side bounding box, so the backend clips to `lat_lim` / `lon_lim` client-side.

In [ ]:
import datetime as dt
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('gdacs_output')
OUT_DIR.mkdir(exist_ok=True)

# A recent 120-day window over the Asia-Pacific basin (active for both
# seismicity and tropical cyclones).
TODAY = dt.date.today()
START = (TODAY - dt.timedelta(days=120)).strftime('%Y-%m-%d')
END = TODAY.strftime('%Y-%m-%d')
LAT_LIM = [-15, 45]
LON_LIM = [90, 180]
print('window:', START, 'to', END, '| box lat', LAT_LIM, 'lon', LON_LIM)

In [ ]:
alerts = None
try:
    alerts = EarthLens(
        variables=['EQ', 'TC'],
        data_source='gdacs',
        start=START,
        end=END,
        lat_lim=LAT_LIM,
        lon_lim=LON_LIM,
        alert_level=['Green', 'Orange', 'Red'],
        path=str(OUT_DIR),
    ).download()
    print(f'{len(alerts)} alerts in box')
    print(alerts['hazard_type'].value_counts().to_dict())
except Exception as exc:
    print(f'live GDACS query skipped (offline?): {type(exc).__name__}: {exc}')

## Overlay the two hazards

Earthquakes are drawn as circles and cyclones as triangles; both are coloured by alert level. Either hazard may be empty in a given window — the plot handles that gracefully.

In [ ]:
if alerts is not None and len(alerts):
    import matplotlib.pyplot as plt

    colours = {'Green': 'green', 'Orange': 'orange', 'Red': 'red'}
    markers = {'EQ': 'o', 'TC': '^'}
    fig, ax = plt.subplots(figsize=(11, 7))
    for code, marker in markers.items():
        subset = alerts[alerts['hazard_type'] == code]
        if len(subset):
            subset.plot(
                ax=ax,
                marker=marker,
                color=subset['alert_level'].map(colours).fillna('grey'),
                markersize=45,
                alpha=0.7,
                label=code,
            )
    ax.set_xlim(LON_LIM)
    ax.set_ylim(LAT_LIM)
    ax.set_xlabel('longitude')
    ax.set_ylabel('latitude')
    ax.set_title(f'EQ (circles) + TC (triangles), Asia-Pacific, {START} to {END}')
    ax.legend(title='hazard')
    plt.show()

## Cross-reference via GLIDE

GDACS assigns a [GLIDE](https://glidenumber.net/) identifier to many alerts — a stable key shared with [ReliefWeb](https://reliefweb.int/) and [Copernicus EMS](https://emergency.copernicus.eu/). Alerts that carry one can be joined to those sources for impact and response detail.

In [ ]:
if alerts is not None and len(alerts):
    has_glide = alerts[alerts['glide'].notna() & (alerts['glide'] != '')]
    print(f'{len(has_glide)} of {len(alerts)} alerts carry a GLIDE id')
    cols = ['hazard_type', 'name', 'country', 'alert_level', 'glide']
    display(has_glide[cols].head(15))